# AILS Visualization

This notebook creates publication-quality visualizations for the AILS algorithm.

**Author:** Amr Elshahed  
**Institution:** Universiti Sains Malaysia

---

## Visualizations:

1. **Path Visualization**: Show corridors and paths
2. **Performance Heatmaps**: Grid size vs density analysis
3. **Comparison Charts**: Bar charts and line plots
4. **Corridor Analysis**: Visualize corridor construction
5. **Publication Figures**: Journal-ready figures

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from tqdm import tqdm
import os

# Import AILS core module
from ails_core import (
    AILSPathfinder, AILSConfig, AILSCorridorBuilder,
    GridGenerator, run_benchmark, compute_statistics
)

# Create figures directory
os.makedirs('figures', exist_ok=True)

# Set publication-quality defaults
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Color scheme
COLORS = {
    'A*': '#1f77b4',
    'Dijkstra': '#ff7f0e',
    'BFS': '#2ca02c',
    'Bidirectional A*': '#d62728',
    'AILS-Base': '#9467bd',
    'AILS-Adaptive': '#8c564b'
}

print("Setup complete!")

## 1. Path and Corridor Visualization

In [ ]:
# Generate test grid
grid = GridGenerator.generate_random(100, density=0.25, seed=42)

# Find start and goal
traversable = np.argwhere(grid == 0)
np.random.seed(123)
idx = np.random.choice(len(traversable), 2, replace=False)
start = tuple(traversable[idx[0]])
goal = tuple(traversable[idx[1]])

print(f"Grid size: {grid.shape}")
print(f"Start: {start}")
print(f"Goal: {goal}")

In [ ]:
# Build corridors and find paths
pathfinder = AILSPathfinder(grid)
corridor_builder = AILSCorridorBuilder(grid)

# Get corridors
base_corridor = corridor_builder.build_base_corridor(start, goal)
adaptive_corridor = corridor_builder.build_adaptive_corridor(start, goal)

# Get paths
astar_result = pathfinder.find_path_standard(start, goal, 'astar')
ails_result = pathfinder.find_path_ails(start, goal, strategy='adaptive')

print(f"Base corridor size: {len(base_corridor)}")
print(f"Adaptive corridor size: {len(adaptive_corridor)}")

In [ ]:
def visualize_corridor(grid, corridor, path, start, goal, title, ax):
    """Visualize corridor and path on grid."""
    # Create visualization grid
    vis = np.zeros_like(grid, dtype=float)
    
    # Mark obstacles
    vis[grid == 1] = 0.3  # Dark gray for obstacles
    
    # Mark corridor
    for r, c in corridor:
        if grid[r, c] == 0:
            vis[r, c] = 0.7  # Light for corridor
    
    # Plot
    ax.imshow(vis, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
    
    # Plot path
    if path:
        path_arr = np.array(path)
        ax.plot(path_arr[:, 1], path_arr[:, 0], 'b-', linewidth=2, label='Path')
    
    # Mark start and goal
    ax.scatter(start[1], start[0], c='green', s=150, marker='s', 
               label='Start', zorder=5, edgecolors='white', linewidths=2)
    ax.scatter(goal[1], goal[0], c='red', s=200, marker='*', 
               label='Goal', zorder=5, edgecolors='white', linewidths=2)
    
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.axis('off')

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Standard A* (full grid search)
ax = axes[0]
full_grid_corridor = set((r, c) for r in range(grid.shape[0]) 
                         for c in range(grid.shape[1]) if grid[r, c] == 0)
visualize_corridor(grid, full_grid_corridor, astar_result.path, start, goal,
                  f"A* (Full Grid)\nNodes: {astar_result.nodes_visited}", ax)

# AILS Base Corridor
ax = axes[1]
visualize_corridor(grid, base_corridor, ails_result.path, start, goal,
                  f"AILS-Base\nCorridor: {len(base_corridor)} cells", ax)

# AILS Adaptive Corridor
ax = axes[2]
visualize_corridor(grid, adaptive_corridor, ails_result.path, start, goal,
                  f"AILS-Adaptive\nCorridor: {len(adaptive_corridor)} cells", ax)

plt.tight_layout()
plt.savefig('figures/corridor_comparison.png', dpi=300)
plt.show()

## 2. Performance Heatmap

In [ ]:
# Generate performance heatmap data
GRID_SIZES = [50, 100, 150, 200, 250]
DENSITIES = [0.15, 0.20, 0.25, 0.30, 0.35]
NUM_PAIRS = 30

node_reduction_matrix = np.zeros((len(GRID_SIZES), len(DENSITIES)))
time_reduction_matrix = np.zeros((len(GRID_SIZES), len(DENSITIES)))

print("Generating heatmap data...")

for i, size in enumerate(tqdm(GRID_SIZES, desc="Grid sizes")):
    for j, density in enumerate(DENSITIES):
        grid = GridGenerator.generate_random(size, density, seed=42)
        
        results = run_benchmark(
            grid, num_pairs=NUM_PAIRS, seed=42,
            algorithms=['A*', 'AILS-Adaptive']
        )
        
        stats = compute_statistics(results)
        
        if stats['A*']['nodes_mean'] > 0:
            node_reduction = (1 - stats['AILS-Adaptive']['nodes_mean'] / 
                            stats['A*']['nodes_mean']) * 100
            time_reduction = (1 - stats['AILS-Adaptive']['time_mean'] / 
                            stats['A*']['time_mean']) * 100
            
            node_reduction_matrix[i, j] = node_reduction
            time_reduction_matrix[i, j] = time_reduction

print("Heatmap data generated!")

In [ ]:
# Plot heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Node reduction heatmap
ax = axes[0]
im = ax.imshow(node_reduction_matrix, cmap='RdYlGn', aspect='auto',
               vmin=0, vmax=80)
ax.set_xticks(range(len(DENSITIES)))
ax.set_xticklabels([f"{d*100:.0f}%" for d in DENSITIES])
ax.set_yticks(range(len(GRID_SIZES)))
ax.set_yticklabels([f"{s}x{s}" for s in GRID_SIZES])
ax.set_xlabel('Obstacle Density')
ax.set_ylabel('Grid Size')
ax.set_title('Node Reduction (%) - AILS vs A*')

# Add value annotations
for i in range(len(GRID_SIZES)):
    for j in range(len(DENSITIES)):
        text = ax.text(j, i, f"{node_reduction_matrix[i, j]:.0f}%",
                      ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im, ax=ax, label='Reduction (%)')

# Time reduction heatmap
ax = axes[1]
im = ax.imshow(time_reduction_matrix, cmap='RdYlGn', aspect='auto',
               vmin=-20, vmax=60)
ax.set_xticks(range(len(DENSITIES)))
ax.set_xticklabels([f"{d*100:.0f}%" for d in DENSITIES])
ax.set_yticks(range(len(GRID_SIZES)))
ax.set_yticklabels([f"{s}x{s}" for s in GRID_SIZES])
ax.set_xlabel('Obstacle Density')
ax.set_ylabel('Grid Size')
ax.set_title('Time Reduction (%) - AILS vs A*')

for i in range(len(GRID_SIZES)):
    for j in range(len(DENSITIES)):
        text = ax.text(j, i, f"{time_reduction_matrix[i, j]:.0f}%",
                      ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im, ax=ax, label='Reduction (%)')

plt.tight_layout()
plt.savefig('figures/performance_heatmap.png', dpi=300)
plt.show()

## 3. Scalability Line Plot

In [ ]:
# Generate scalability data
GRID_SIZES = [50, 100, 150, 200, 250, 300, 350, 400]
NUM_PAIRS = 30

scalability_data = {alg: {'sizes': [], 'nodes': [], 'times': []} 
                   for alg in ['A*', 'AILS-Base', 'AILS-Adaptive']}

print("Generating scalability data...")

for size in tqdm(GRID_SIZES, desc="Grid sizes"):
    grid = GridGenerator.generate_random(size, 0.25, seed=42)
    
    results = run_benchmark(
        grid, num_pairs=NUM_PAIRS, seed=42,
        algorithms=['A*', 'AILS-Base', 'AILS-Adaptive']
    )
    
    stats = compute_statistics(results)
    
    for alg in scalability_data.keys():
        scalability_data[alg]['sizes'].append(size)
        scalability_data[alg]['nodes'].append(stats[alg]['nodes_mean'])
        scalability_data[alg]['times'].append(stats[alg]['time_mean'])

print("Scalability data generated!")

In [ ]:
# Plot scalability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Nodes vs Grid Size
ax = axes[0]
for alg in ['A*', 'AILS-Base', 'AILS-Adaptive']:
    ax.plot(scalability_data[alg]['sizes'], scalability_data[alg]['nodes'],
            marker='o', linewidth=2, markersize=8, label=alg, color=COLORS[alg])
ax.set_xlabel('Grid Size')
ax.set_ylabel('Nodes Visited')
ax.set_title('Scalability: Nodes Visited vs Grid Size')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Time vs Grid Size
ax = axes[1]
for alg in ['A*', 'AILS-Base', 'AILS-Adaptive']:
    ax.plot(scalability_data[alg]['sizes'], scalability_data[alg]['times'],
            marker='o', linewidth=2, markersize=8, label=alg, color=COLORS[alg])
ax.set_xlabel('Grid Size')
ax.set_ylabel('Time (ms)')
ax.set_title('Scalability: Execution Time vs Grid Size')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('figures/scalability_plot.png', dpi=300)
plt.show()

## 4. Algorithm Comparison Bar Chart

In [ ]:
# Generate comparison data
grid = GridGenerator.generate_random(200, 0.25, seed=42)

results = run_benchmark(
    grid, num_pairs=100, seed=42,
    algorithms=['A*', 'Dijkstra', 'BFS', 'Bidirectional A*', 'AILS-Base', 'AILS-Adaptive']
)

stats = compute_statistics(results)

# Prepare data
algorithms = list(stats.keys())
nodes = [stats[a]['nodes_mean'] for a in algorithms]
times = [stats[a]['time_mean'] for a in algorithms]
nodes_std = [stats[a]['nodes_std'] for a in algorithms]
times_std = [stats[a]['time_std'] for a in algorithms]

In [ ]:
# Create grouped bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x = np.arange(len(algorithms))
width = 0.6
colors = [COLORS[a] for a in algorithms]

# Nodes comparison
ax = axes[0]
bars = ax.bar(x, nodes, width, yerr=nodes_std, color=colors, capsize=5, alpha=0.8)
ax.set_ylabel('Nodes Visited')
ax.set_title('Algorithm Comparison: Nodes Visited')
ax.set_xticks(x)
ax.set_xticklabels(algorithms, rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, nodes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(nodes)*0.02,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

# Time comparison
ax = axes[1]
bars = ax.bar(x, times, width, yerr=times_std, color=colors, capsize=5, alpha=0.8)
ax.set_ylabel('Time (ms)')
ax.set_title('Algorithm Comparison: Execution Time')
ax.set_xticks(x)
ax.set_xticklabels(algorithms, rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/algorithm_comparison.png', dpi=300)
plt.show()

## 5. Obstacle Pattern Visualization

In [ ]:
# Generate different pattern grids
patterns = {
    'Random': GridGenerator.generate_random(100, 0.25, seed=42),
    'Clustered': GridGenerator.generate_clustered(100, 0.25, 15, seed=42),
    'Maze': GridGenerator.generate_maze(101, seed=42),
    'Room': GridGenerator.generate_room(100, num_rooms=6, seed=42),
    'Open': GridGenerator.generate_open(100, 0.1, seed=42)
}

# Visualize patterns
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, grid) in zip(axes, patterns.items()):
    ax.imshow(grid, cmap='binary', origin='upper')
    density = np.mean(grid) * 100
    ax.set_title(f"{name}\n({density:.1f}% obstacles)")
    ax.axis('off')

plt.tight_layout()
plt.savefig('figures/obstacle_patterns.png', dpi=300)
plt.show()

In [ ]:
# Pattern performance comparison
pattern_results = []

print("Analyzing pattern performance...")

for pattern_name, grid in tqdm(patterns.items(), desc="Patterns"):
    results = run_benchmark(
        grid, num_pairs=50, seed=42,
        algorithms=['A*', 'AILS-Adaptive']
    )
    stats = compute_statistics(results)
    
    if stats['A*']['nodes_mean'] > 0:
        reduction = (1 - stats['AILS-Adaptive']['nodes_mean'] / 
                    stats['A*']['nodes_mean']) * 100
    else:
        reduction = 0
    
    pattern_results.append({
        'pattern': pattern_name,
        'astar_nodes': stats['A*']['nodes_mean'],
        'ails_nodes': stats['AILS-Adaptive']['nodes_mean'],
        'reduction': reduction
    })

df_patterns = pd.DataFrame(pattern_results)
print(df_patterns)

In [ ]:
# Plot pattern comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(df_patterns))
width = 0.35

bars1 = ax.bar(x - width/2, df_patterns['astar_nodes'], width, 
               label='A*', color=COLORS['A*'], alpha=0.8)
bars2 = ax.bar(x + width/2, df_patterns['ails_nodes'], width,
               label='AILS-Adaptive', color=COLORS['AILS-Adaptive'], alpha=0.8)

ax.set_ylabel('Nodes Visited')
ax.set_title('Performance Across Different Obstacle Patterns')
ax.set_xticks(x)
ax.set_xticklabels(df_patterns['pattern'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add reduction labels
for i, (b1, b2, red) in enumerate(zip(bars1, bars2, df_patterns['reduction'])):
    ax.annotate(f'{red:.0f}% reduction',
                xy=(i, max(b1.get_height(), b2.get_height())),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=10, color='green')

plt.tight_layout()
plt.savefig('figures/pattern_comparison.png', dpi=300)
plt.show()

## 6. Corridor Construction Animation (Static Frames)

In [ ]:
# Visualize corridor construction steps
grid = GridGenerator.generate_random(80, 0.25, seed=42)

# Find start and goal
traversable = np.argwhere(grid == 0)
np.random.seed(456)
idx = np.random.choice(len(traversable), 2, replace=False)
start = tuple(traversable[idx[0]])
goal = tuple(traversable[idx[1]])

# Build corridor builder
config = AILSConfig(r_min=3, r_max=12)
builder = AILSCorridorBuilder(grid, config)

# Get Bresenham line
line_points = builder.bresenham_line(start, goal)

print(f"Line points: {len(line_points)}")
print(f"Start: {start}, Goal: {goal}")

In [ ]:
# Create corridor construction visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Step 1: Grid with obstacles
ax = axes[0]
ax.imshow(grid, cmap='binary', origin='upper')
ax.scatter(start[1], start[0], c='green', s=100, marker='s', zorder=5)
ax.scatter(goal[1], goal[0], c='red', s=150, marker='*', zorder=5)
ax.set_title('Step 1: Grid Environment')
ax.axis('off')

# Step 2: Bresenham line
ax = axes[1]
ax.imshow(grid, cmap='binary', origin='upper')
line_arr = np.array(line_points)
ax.plot(line_arr[:, 1], line_arr[:, 0], 'b-', linewidth=2, alpha=0.7)
ax.scatter(start[1], start[0], c='green', s=100, marker='s', zorder=5)
ax.scatter(goal[1], goal[0], c='red', s=150, marker='*', zorder=5)
ax.set_title('Step 2: Bresenham Line')
ax.axis('off')

# Step 3: Local density computation
ax = axes[2]
density_map = np.zeros_like(grid, dtype=float)
for r in range(grid.shape[0]):
    for c in range(grid.shape[1]):
        density_map[r, c] = builder.compute_local_density((r, c))
im = ax.imshow(density_map, cmap='YlOrRd', origin='upper')
ax.plot(line_arr[:, 1], line_arr[:, 0], 'b-', linewidth=2, alpha=0.7)
ax.scatter(start[1], start[0], c='green', s=100, marker='s', zorder=5)
ax.scatter(goal[1], goal[0], c='red', s=150, marker='*', zorder=5)
ax.set_title('Step 3: Local Density Map')
ax.axis('off')
plt.colorbar(im, ax=ax, label='Density')

# Step 4: Base corridor
ax = axes[3]
base_corridor = builder.build_base_corridor(start, goal)
vis = np.zeros_like(grid, dtype=float)
vis[grid == 1] = 0.3
for r, c in base_corridor:
    vis[r, c] = 0.7
ax.imshow(vis, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
ax.scatter(start[1], start[0], c='green', s=100, marker='s', zorder=5)
ax.scatter(goal[1], goal[0], c='red', s=150, marker='*', zorder=5)
ax.set_title(f'Step 4: Base Corridor (r_min={config.r_min})')
ax.axis('off')

# Step 5: Adaptive corridor
ax = axes[4]
adaptive_corridor = builder.build_adaptive_corridor(start, goal)
vis = np.zeros_like(grid, dtype=float)
vis[grid == 1] = 0.3
for r, c in adaptive_corridor:
    vis[r, c] = 0.7
ax.imshow(vis, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
ax.scatter(start[1], start[0], c='green', s=100, marker='s', zorder=5)
ax.scatter(goal[1], goal[0], c='red', s=150, marker='*', zorder=5)
ax.set_title('Step 5: Adaptive Corridor')
ax.axis('off')

# Step 6: Final path
ax = axes[5]
pathfinder = AILSPathfinder(grid, config)
result = pathfinder.find_path_ails(start, goal, strategy='adaptive')
vis = np.zeros_like(grid, dtype=float)
vis[grid == 1] = 0.3
for r, c in adaptive_corridor:
    vis[r, c] = 0.7
ax.imshow(vis, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
if result.path:
    path_arr = np.array(result.path)
    ax.plot(path_arr[:, 1], path_arr[:, 0], 'b-', linewidth=3)
ax.scatter(start[1], start[0], c='green', s=100, marker='s', zorder=5)
ax.scatter(goal[1], goal[0], c='red', s=150, marker='*', zorder=5)
ax.set_title(f'Step 6: Final Path (Nodes: {result.nodes_visited})')
ax.axis('off')

plt.tight_layout()
plt.savefig('figures/corridor_construction_steps.png', dpi=300)
plt.show()

## 7. Publication-Ready Summary Figure

In [ ]:
# Create comprehensive summary figure
fig = plt.figure(figsize=(16, 12))

# Grid layout
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# (a) Corridor visualization
ax1 = fig.add_subplot(gs[0, 0])
grid = GridGenerator.generate_random(80, 0.25, seed=42)
traversable = np.argwhere(grid == 0)
np.random.seed(789)
idx = np.random.choice(len(traversable), 2, replace=False)
start, goal = tuple(traversable[idx[0]]), tuple(traversable[idx[1]])

builder = AILSCorridorBuilder(grid)
corridor = builder.build_adaptive_corridor(start, goal)
pathfinder = AILSPathfinder(grid)
result = pathfinder.find_path_ails(start, goal)

vis = np.zeros_like(grid, dtype=float)
vis[grid == 1] = 0.3
for r, c in corridor:
    vis[r, c] = 0.7
ax1.imshow(vis, cmap='RdYlGn', vmin=0, vmax=1, origin='upper')
if result.path:
    path_arr = np.array(result.path)
    ax1.plot(path_arr[:, 1], path_arr[:, 0], 'b-', linewidth=2)
ax1.scatter(start[1], start[0], c='green', s=80, marker='s', zorder=5)
ax1.scatter(goal[1], goal[0], c='red', s=100, marker='*', zorder=5)
ax1.set_title('(a) AILS Corridor and Path')
ax1.axis('off')

# (b) Scalability plot
ax2 = fig.add_subplot(gs[0, 1])
for alg in ['A*', 'AILS-Adaptive']:
    ax2.plot(scalability_data[alg]['sizes'], scalability_data[alg]['nodes'],
            marker='o', linewidth=2, label=alg, color=COLORS[alg])
ax2.set_xlabel('Grid Size')
ax2.set_ylabel('Nodes Visited')
ax2.set_title('(b) Scalability Analysis')
ax2.legend()
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

# (c) Node reduction heatmap
ax3 = fig.add_subplot(gs[0, 2])
im = ax3.imshow(node_reduction_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=80)
ax3.set_xticks(range(len(DENSITIES)))
ax3.set_xticklabels([f"{d*100:.0f}%" for d in DENSITIES])
ax3.set_yticks(range(len(GRID_SIZES)))
ax3.set_yticklabels([f"{s}" for s in GRID_SIZES])
ax3.set_xlabel('Density')
ax3.set_ylabel('Size')
ax3.set_title('(c) Node Reduction (%)')
plt.colorbar(im, ax=ax3)

# (d) Algorithm comparison bars
ax4 = fig.add_subplot(gs[1, :])
x = np.arange(len(algorithms))
bars = ax4.bar(x, nodes, color=[COLORS[a] for a in algorithms], alpha=0.8)
ax4.set_ylabel('Nodes Visited')
ax4.set_title('(d) Algorithm Comparison (200x200 grid, 25% density)')
ax4.set_xticks(x)
ax4.set_xticklabels(algorithms)
ax4.grid(True, alpha=0.3, axis='y')

# (e) Pattern comparison
ax5 = fig.add_subplot(gs[2, 0:2])
x = np.arange(len(df_patterns))
width = 0.35
ax5.bar(x - width/2, df_patterns['astar_nodes'], width, 
       label='A*', color=COLORS['A*'], alpha=0.8)
ax5.bar(x + width/2, df_patterns['ails_nodes'], width,
       label='AILS-Adaptive', color=COLORS['AILS-Adaptive'], alpha=0.8)
ax5.set_ylabel('Nodes Visited')
ax5.set_title('(e) Performance by Obstacle Pattern')
ax5.set_xticks(x)
ax5.set_xticklabels(df_patterns['pattern'])
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

# (f) Improvement summary
ax6 = fig.add_subplot(gs[2, 2])
astar_baseline = stats['A*']['nodes_mean']
improvements = [(a, (1 - stats[a]['nodes_mean']/astar_baseline)*100) 
                for a in ['AILS-Base', 'AILS-Adaptive']]
ax6.barh([i[0] for i in improvements], [i[1] for i in improvements],
         color=[COLORS[i[0]] for i in improvements], alpha=0.8)
ax6.set_xlabel('Node Reduction vs A* (%)')
ax6.set_title('(f) AILS Improvement')
ax6.grid(True, alpha=0.3, axis='x')
for i, (name, val) in enumerate(improvements):
    ax6.text(val + 1, i, f'{val:.1f}%', va='center', fontsize=11)

plt.suptitle('AILS: Adaptive Incremental Line Search - Performance Summary', 
             fontsize=16, fontweight='bold', y=1.02)

plt.savefig('figures/publication_summary.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save All Figures List

In [ ]:
# List all generated figures
print("\nGenerated Figures:")
print("="*50)

for f in sorted(os.listdir('figures')):
    filepath = os.path.join('figures', f)
    size = os.path.getsize(filepath) / 1024  # KB
    print(f"  {f:<40} ({size:.1f} KB)")

print("\nAll figures are saved in the 'figures/' directory.")

## Summary

This notebook created publication-quality visualizations including:

1. **Corridor Visualization**: Shows how AILS constrains search space
2. **Performance Heatmaps**: Grid size vs density analysis
3. **Scalability Plots**: Performance as grid size increases
4. **Algorithm Comparison**: Bar charts comparing all methods
5. **Pattern Analysis**: Performance across different obstacle patterns
6. **Construction Steps**: Step-by-step corridor building visualization
7. **Summary Figure**: Comprehensive publication-ready figure

All figures are saved in high resolution (300 DPI) suitable for journal submission.